In [0]:
from pyspark.sql import Row

# --- BU Group definitions ---
BU_GROUPS = {
    'COMMERCIALS': [
        'COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL',
        'COMNA-NORTH AMERICA', 'COMNN-NETHERLANDS', 'COMON-OCEANIA',
        'COMSE-BELGIUM, LUXEMBOURG', 'COMSPB-SPAIN, PORTUGAL, BRAZIL',
        'COMUI-UK & IRELAND', 'FRANC-FRANCE', 'ITALY-ITALY', 'CREDIT INSURANCE'
    ],
    'RISK': [
        'RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE', 'RISK2-RS2-BELGIUM, LUX, FRANCE & ITA',
        'RISK3-RS3-NLD, APAC & AIM', 'RISK4-RS4-NORTH EUROPE, CIS & ACS',
        'RISK4-RS4-NORTH EUROPE, CIS & SP', 'RISK5-RS5-NAFTA',
        'RISK1-RS1-DEU, AUT and CHE', 'RISK1-RS1-Europe, Russia & CIS',
        'RISK2-RS2-NLD, Belux, FRA, Africa & ISR', 'RISK3-RS3-Asia and Oceania',
        'RISK7-RS7-Spain, Portugal, Andorra'
    ],
    'FINANCE': [
        'FINPM-FINANCE PROGRAM MANAGEMENT', 'GFO-GROUP FINANCE OPERATIONS',
        'GFC-GROUP FINANCE AND CONTROL', 'COFIN-CORPORATE FINANCE & TAX',
        'Group Finance', 'Finance', 'Finance & Control'
    ],
    'Unidentified': [
        'Unidentified', 'LEFT Organization'
    ],
}

CLUSTER_NAMES = {
    0: 'Commercial & Business Operations',
    1: 'Financial Reporting & Movement Analysis',
    2: 'Declarations & Policy Group KPI Monitoring',
    3: 'Credit Limits & Engagements Management',
    4: 'Claims Management & Loss Reporting',
    5: 'Core Reference Data & Back-Office Operations',
    6: 'Policy Details, Contracts & Customer Portfolio Analytics',
    7: 'User Access, Authorizations & Sales Pipeline',
    8: 'Global Policy Portfolio & Administration',
    9: 'Premium, Tax & Invoicing Automation',
}

# === Write BU_Grouping table ===
bu_rows = []
for group_name, bus in BU_GROUPS.items():
    for bu in bus:
        bu_rows.append(Row(BU_Group=group_name, BU=bu))

df_bu_grouping = spark.createDataFrame(bu_rows)
df_bu_grouping.write.mode("overwrite").saveAsTable(
    "dataplatform01_central_dev_catalog_01.custom_sap_bo.BU_Grouping"
)
print(f"BU_Grouping table created with {len(bu_rows)} rows")
display(df_bu_grouping)

# === Write cluster_names table ===
cluster_rows = [Row(cluster_id=k, cluster_name=v) for k, v in CLUSTER_NAMES.items()]

df_cluster_names = spark.createDataFrame(cluster_rows)
df_cluster_names.write.mode("overwrite").saveAsTable(
    "dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_names"
)
print(f"cluster_names table created with {len(cluster_rows)} rows")
display(df_cluster_names)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Cluster name mappings ---
CLUSTER_NAMES = {
    0: 'Commercial & Business Operations',
    1: 'Financial Reporting & Movement Analysis',
    2: 'Declarations & Policy Group KPI Monitoring',
    3: 'Credit Limits & Engagements Management',
    4: 'Claims Management & Loss Reporting',
    5: 'Core Reference Data & Back-Office Operations',
    6: 'Policy Details, Contracts & Customer Portfolio Analytics',
    7: 'User Access, Authorizations & Sales Pipeline',
    8: 'Global Policy Portfolio & Administration',
    9: 'Premium, Tax & Invoicing Automation',
}

# --- BU Group definitions ---
BU_GROUPS = {
    'COMMERCIALS': [
        'COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL',
        'COMNA-NORTH AMERICA', 'COMNN-NETHERLANDS', 'COMON-OCEANIA',
        'COMSE-BELGIUM, LUXEMBOURG', 'COMSPB-SPAIN, PORTUGAL, BRAZIL',
        'COMUI-UK & IRELAND', 'FRANC-FRANCE', 'ITALY-ITALY', 'CREDIT INSURANCE'
    ],
    'RISK': [
        'RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE', 'RISK2-RS2-BELGIUM, LUX, FRANCE & ITA',
        'RISK3-RS3-NLD, APAC & AIM', 'RISK4-RS4-NORTH EUROPE, CIS & ACS',
        'RISK4-RS4-NORTH EUROPE, CIS & SP', 'RISK5-RS5-NAFTA',
        'RISK1-RS1-DEU, AUT and CHE', 'RISK1-RS1-Europe, Russia & CIS',
        'RISK2-RS2-NLD, Belux, FRA, Africa & ISR', 'RISK3-RS3-Asia and Oceania',
        'RISK7-RS7-Spain, Portugal, Andorra'
    ],
    'FINANCE': [
        'FINPM-FINANCE PROGRAM MANAGEMENT', 'GFO-GROUP FINANCE OPERATIONS',
        'GFC-GROUP FINANCE AND CONTROL', 'COFIN-CORPORATE FINANCE & TAX',
        'Group Finance', 'Finance', 'Finance & Control'
    ],
    'Unidentified': [
        'Unidentified', 'LEFT Organization'
    ],
}

# Build CASE WHEN SQL — NULL BU maps to Unidentified, unmapped BUs use actual name
case_parts = ["WHEN BU IS NULL THEN 'Unidentified'"]
for group_name, bus in BU_GROUPS.items():
    bu_list = ", ".join([f"'{b.upper()}'" for b in bus])
    case_parts.append(f"WHEN UPPER(BU) IN ({bu_list}) THEN '{group_name}'")
case_sql = "CASE " + " ".join(case_parts) + " ELSE UPPER(BU) END"

# Query: distinct documents per cluster per BU group (exclude unclustered)
query = f"""
SELECT
  CAST(CAST(cluster AS INT) AS STRING) AS cluster,
  {case_sql} AS BU_Group,
  COUNT(DISTINCT Doc_ID) AS doc_count
FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
WHERE cluster IS NOT NULL
GROUP BY cluster, BU_Group
ORDER BY cluster, BU_Group
"""

pdf = spark.sql(query).toPandas()

# ============================================================
# CHART 1: Cluster on x-axis, BU Group as legend (stacked)
# ============================================================
pivot_df = pdf.pivot_table(index='cluster', columns='BU_Group', values='doc_count', fill_value=0)
pivot_df = pivot_df.loc[sorted(pivot_df.index, key=lambda x: int(x))]

def cluster_label(c):
    try:
        return f"{c}: {CLUSTER_NAMES[int(c)]}"
    except (ValueError, KeyError):
        return c

pivot_df.index = [cluster_label(c) for c in pivot_df.index]

defined_groups = ['COMMERCIALS', 'RISK', 'FINANCE', 'Unidentified']
other_cols = sorted([c for c in pivot_df.columns if c not in defined_groups])
ordered_cols = [g for g in defined_groups if g in pivot_df.columns] + other_cols
pivot_df = pivot_df[ordered_cols]

fig, ax = plt.subplots(figsize=(16, 7))
group_colors = {'COMMERCIALS': '#2196F3', 'RISK': '#FF9800', 'FINANCE': '#4CAF50', 'Unidentified': '#9E9E9E'}
cmap = plt.colormaps.get_cmap('tab20')
for i, col in enumerate(other_cols):
    group_colors[col] = cmap(i / max(len(other_cols), 1))
bar_colors = [group_colors.get(col, '#9E9E9E') for col in pivot_df.columns]

pivot_df.plot(kind='bar', stacked=True, ax=ax, color=bar_colors, width=0.6)
ax.set_title('Documents per Cluster by BU Group (Stacked)', fontsize=14, fontweight='bold')
ax.set_xlabel('Cluster', fontsize=11)
ax.set_ylabel('Distinct Documents', fontsize=11)
ax.legend(title='BU Group', fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
for container in ax.containers:
    labels = [f'{int(v.get_height())}' if v.get_height() > 30 else '' for v in container]
    ax.bar_label(container, labels=labels, label_type='center', fontsize=6, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================
# CHART 2: BU Group on x-axis, Cluster as legend (stacked)
# ============================================================
pivot_df2 = pdf.pivot_table(index='BU_Group', columns='cluster', values='doc_count', fill_value=0)

# Order clusters numerically
cluster_order = sorted(pivot_df2.columns, key=lambda x: int(x))
pivot_df2 = pivot_df2[cluster_order]

# Rename cluster columns to include names
pivot_df2.columns = [f"{c}: {CLUSTER_NAMES.get(int(c), c)}" for c in pivot_df2.columns]

# Sort BU groups: defined groups first, then others by total descending
bu_order = [g for g in defined_groups if g in pivot_df2.index]
other_bus = [b for b in pivot_df2.index if b not in defined_groups]
other_bus_sorted = pivot_df2.loc[other_bus].sum(axis=1).sort_values(ascending=False).index.tolist()
pivot_df2 = pivot_df2.loc[bu_order + other_bus_sorted]

fig2, ax2 = plt.subplots(figsize=(16, 7))
# Cluster color palette
cluster_cmap = plt.colormaps.get_cmap('tab10')
cluster_colors = [cluster_cmap(int(c) / 10) for c in cluster_order]

pivot_df2.plot(kind='bar', stacked=True, ax=ax2, color=cluster_colors, width=0.6)
ax2.set_title('Documents per BU Group by Cluster (Stacked)', fontsize=14, fontweight='bold')
ax2.set_xlabel('BU Group', fontsize=11)
ax2.set_ylabel('Distinct Documents', fontsize=11)
ax2.legend(title='Cluster', fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
for container in ax2.containers:
    labels = [f'{int(v.get_height())}' if v.get_height() > 30 else '' for v in container]
    ax2.bar_label(container, labels=labels, label_type='center', fontsize=6, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

In [0]:
%sql
SELECT
  COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster,
  CASE WHEN BU IS NULL THEN 'Unidentified' WHEN UPPER(BU) IN ('COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL', 'COMNA-NORTH AMERICA', 'COMNN-NETHERLANDS', 'COMON-OCEANIA', 'COMSE-BELGIUM, LUXEMBOURG', 'COMSPB-SPAIN, PORTUGAL, BRAZIL', 'COMUI-UK & IRELAND', 'FRANC-FRANCE', 'ITALY-ITALY', 'CREDIT INSURANCE') THEN 'COMMERCIALS' WHEN UPPER(BU) IN ('RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE', 'RISK2-RS2-BELGIUM, LUX, FRANCE & ITA', 'RISK3-RS3-NLD, APAC & AIM', 'RISK4-RS4-NORTH EUROPE, CIS & ACS', 'RISK4-RS4-NORTH EUROPE, CIS & SP', 'RISK5-RS5-NAFTA', 'RISK1-RS1-DEU, AUT AND CHE', 'RISK1-RS1-EUROPE, RUSSIA & CIS', 'RISK2-RS2-NLD, BELUX, FRA, AFRICA & ISR', 'RISK3-RS3-ASIA AND OCEANIA', 'RISK7-RS7-SPAIN, PORTUGAL, ANDORRA') THEN 'RISK' WHEN UPPER(BU) IN ('FINPM-FINANCE PROGRAM MANAGEMENT', 'GFO-GROUP FINANCE OPERATIONS', 'GFC-GROUP FINANCE AND CONTROL', 'COFIN-CORPORATE FINANCE & TAX', 'GROUP FINANCE', 'FINANCE', 'FINANCE & CONTROL') THEN 'FINANCE' WHEN UPPER(BU) IN ('UNIDENTIFIED', 'LEFT ORGANIZATION') THEN 'Unidentified' ELSE UPPER(BU) END AS BU_Group,
  COUNT(DISTINCT Doc_ID) AS doc_count
FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis as ca1
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as wa1
    on upper(trim(wa1.WEBI_Doc_ID))=upper(trim(ca1.Doc_ID))
where wa1.WEBI_Doc_ID is not null
GROUP BY cluster, BU_Group
ORDER BY cluster, BU_Group

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.BU_Grouping
--  dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_names